# 01 — Extracting a Persona Vector

**Goal:** Given a trait, extract a linear direction in activation space that
represents it. This is the conceptual heart of the paper.

**Pipeline** (paper Section 2.2, Figure 2):

1. For a trait, assemble contrastive **system prompts** (5 positive, 5 negative)
   and 20 extraction **questions**.
2. Under each (polarity × system_prompt × question), generate 10 rollouts.
3. **Judge** each rollout 0–100 for trait expression. Keep only positive rollouts
   scoring >50 and negative rollouts scoring <50.
4. For each layer, average residual-stream activations over the response tokens.
5. **Persona vector** = mean(positive responses) − mean(negative responses), per
   layer.

The library handles steps 1–3 for you (rollouts are cached). Your job is step 5:
the diff-of-means formula — and then choosing which layer to use.


## Load artifacts + model


In [ ]:
from persona_lib import (
    ActivationModel, ChatPrompt, steering,
    run_extraction, save_extraction, load_extraction,
    load_trait, score_trait,
)
import numpy as np
import matplotlib.pyplot as plt

TRAIT = 'evil'   # change to 'sycophancy' or 'hallucination' after this works
artifacts = load_trait(TRAIT)

print("description:\n", artifacts.description[:500], "…")
print("\n# system prompt pair 0 (positive):\n", artifacts.system_prompts[0]['positive'])
print("\n# system prompt pair 0 (negative):\n", artifacts.system_prompts[0]['negative'])
print("\nsample extraction question:", artifacts.extraction_questions[0])


In [ ]:
model = ActivationModel("Qwen/Qwen2.5-7B-Instruct")
print("num_layers:", model.num_layers, "hidden_dim:", model.hidden_dim)


## Run the pipeline

`run_extraction` generates rollouts (or loads them from cache if they exist),
judges them, filters by threshold, and computes the diff-of-means per layer. On a
single A100 this takes ~15–30 min the first time; cached runs are ~2 min.


In [ ]:
result = run_extraction(model, TRAIT, n_rollouts=10, judge_threshold=50)
print("vectors_by_layer shape:", result.vectors_by_layer.shape)
print("diagnostics:", result.diagnostics)
save_extraction(result)   # → data/cached/vectors/<trait>/


## TODO 1 — Inspect the diff-of-means yourself

`run_extraction` already computed vectors for you. As a sanity check, recompute
the vector for one layer directly from cached rollouts to make sure you
understand the formula. This is the ENTIRE conceptual crux of the paper — don't
skip it.

The formula, paper Section 2.2:

$$v_\ell = \frac{1}{|P|} \sum_{i \in P} h_\ell^{(i)} - \frac{1}{|N|} \sum_{j \in N} h_\ell^{(j)}$$

where $P$ is the set of filtered positive responses, $N$ is the filtered negative
responses, and $h_\ell$ is the mean residual-stream activation at layer $\ell$
over response tokens.


In [ ]:
from persona_lib import get_cached_rollouts

rollouts = get_cached_rollouts(TRAIT, model.model_id)
pos = [r for r in rollouts if r['polarity'] == 'positive' and r['trait_score'] > 50]
neg = [r for r in rollouts if r['polarity'] == 'negative' and r['trait_score'] < 50]
print(f"positives kept: {len(pos)}, negatives kept: {len(neg)}")


In [ ]:
LAYER = 15   # paper uses mid layers; you'll pick the best one below

def to_chatprompt(row):
    return ChatPrompt(user=row['question'], system=row['system'], assistant=row['response'])

pos_acts = model.collect_activations([to_chatprompt(r) for r in pos], position='response_mean', layers=[LAYER])
neg_acts = model.collect_activations([to_chatprompt(r) for r in neg], position='response_mean', layers=[LAYER])

# TODO: compute v_layer as mean(pos_acts) - mean(neg_acts) along the prompt axis.
# Both arrays have shape (n_responses, 1, hidden_dim). Result should have shape (hidden_dim,).
v_layer = ...  # YOUR CODE HERE

assert v_layer.shape == (model.hidden_dim,), f"got {v_layer.shape}"
# sanity: the extracted vector should match what run_extraction computed for this layer
np.testing.assert_allclose(v_layer, result.vectors_by_layer[LAYER], rtol=1e-4)
print("✓ diff-of-means matches pipeline output")


## TODO 2 — Choose the best layer via a steering sweep

You have one candidate vector *per* layer. The paper (Appendix B.4) picks the
most informative layer by checking steering effectiveness: for a few candidate
layers, generate responses with `h_ℓ ← h_ℓ + α·v_ℓ` applied during generation,
judge them, and pick the layer where steering most increases trait expression.

Fill in the sweep below. Keep `α` at around `1.5` and use ~5 eval questions.


In [ ]:
CANDIDATE_LAYERS = list(range(4, model.num_layers - 2, 2))  # every 2 layers
ALPHA = 1.5
EVAL_Qs = [ChatPrompt(user=q) for q in artifacts.eval_questions[:5]]

layer_scores = {}
for L in CANDIDATE_LAYERS:
    v = result.vectors_by_layer[L]
    # TODO: Under `steering(model, v, layer=L, alpha=ALPHA)`, generate one response per
    # eval question (temperature=0.7, max_new_tokens=200), score each with
    # `score_trait(resp, TRAIT)`, then take the mean. Store in layer_scores[L].
    layer_scores[L] = ...  # YOUR CODE HERE
    print(f"layer {L:>2}: mean trait score = {layer_scores[L]:.1f}")

best_layer = max(layer_scores, key=layer_scores.get)
print(f"\n★ best layer = {best_layer}")
result.best_layer = best_layer
save_extraction(result)


## Visualize the layer sweep


In [ ]:
from persona_lib.viz import plot_layer_sweep

layers = sorted(layer_scores)
plot_layer_sweep(layers, {ALPHA: [layer_scores[L] for L in layers]},
                 title=f"Steering effectiveness by layer — {TRAIT}")
plt.show()


## Analysis (write ~150 words in this cell)

In your own words, address:

1. Which layer did you pick as "best"? Does it align with the paper's finding
   that mid layers (~15–20 for 7B models) carry trait-specific information?
2. Why do very early and very late layers give weaker steering? (Hint: think
   about what those layers represent.)
3. What would you expect to happen if you used `α = 10.0`? Test it — does the
   trait score keep rising, or does the model's output degrade first?


## What's next

- `02_monitoring.ipynb`: use your vector to *predict* trait expression from
  prompt activations alone (no generation needed).
- `03_inference_steering.ipynb`: causally validate with a full α sweep.
